In [ ]:
# ============================================================
# CELL 1: DATASET & DATALOADER CHO UNET TỪ KVASIR_YOLO_SEG_BG20
# ============================================================
import os, glob, cv2, torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

DATASET_ROOT = "/kaggle/input/datasets/okeydokey0805/kvasir-yolo-seg-bg20/Kvasir_YOLO_SEG_BG20"

class KvasirBG20UNetDataset(Dataset):
    def __init__(self, root_dir, split="train", img_size=(352, 352)):
        self.img_dir = os.path.join(root_dir, "images", split)
        self.label_dir = os.path.join(root_dir, "labels", split)
        self.img_files = sorted(glob.glob(os.path.join(self.img_dir, "*.jpg")))
        self.img_size = img_size
        assert len(self.img_files) > 0, f"Không tìm thấy ảnh tại: {self.img_dir}"
        
    def __len__(self):
        return len(self.img_files)
        
    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        stem = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.label_dir, stem + ".txt")
        
        # 1. Đọc ảnh RGB
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # 2. Tạo Mask nhị phân on-the-fly từ polygon hoặc ảnh nền
        mask = np.zeros((h, w), dtype=np.uint8)
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) > 5: # Định dạng YOLO seg polygon
                        coords = np.array([float(x) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
                        coords[:, 0] *= w
                        coords[:, 1] *= h
                        pts = coords.astype(np.int32)
                        cv2.fillPoly(mask, [pts], 255)
                        
        # 3. Resize kích thước chuẩn (mặc định 352x352 chuẩn polyp benchmark)
        img = cv2.resize(img, self.img_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
        
        # 4. Chuẩn hóa về Tensor [0.0, 1.0]
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        mask_tensor = torch.tensor(mask > 127, dtype=torch.float32).unsqueeze(0) # Shape: (1, H, W)
        
        return img_tensor, mask_tensor

# Khởi tạo kiểm tra nhanh
train_ds = KvasirBG20UNetDataset(DATASET_ROOT, split="train", img_size=(352, 352))
val_ds = KvasirBG20UNetDataset(DATASET_ROOT, split="val", img_size=(352, 352))

print(f"✅ Đã nạp thành công Dataset:")
print(f"   - Tập Train: {len(train_ds)} ảnh (880 polyp + 160 nền)")
print(f"   - Tập Val  : {len(val_ds)} ảnh (120 polyp + 40 nền)")


In [ ]:
# ============================================================
# CELL 2: KHÓA SEED 9, KHỞI TẠO UNET & HUẤN LUYỆN ĐỐI CHUẨN
# ============================================================
import os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# 1. SIÊU THAM SỐ 
SEED = 19
EPOCHS = 100
BATCH_SIZE = 8
LR = 1e-3
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "/kaggle/working/Unet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. KHÓA TẤT ĐỊNH HỆ THỐNG
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# 3. KIẾN TRÚC MÔ HÌNH UNET THUẦN PYTORCH
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.inc = DoubleConv(in_ch, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        
        self.up1 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv_up1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv_up2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv_up3 = DoubleConv(128, 64)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv_up4 = DoubleConv(64, 32)
        
        self.outc = nn.Conv2d(32, out_ch, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5)
        x = self.conv_up1(torch.cat([x, x4], dim=1))
        x = self.up2(x)
        x = self.conv_up2(torch.cat([x, x3], dim=1))
        x = self.up3(x)
        x = self.conv_up3(torch.cat([x, x2], dim=1))
        x = self.up4(x)
        x = self.conv_up4(torch.cat([x, x1], dim=1))
        return self.outc(x)

# 4. HÀM LOSS BCE + DICE TỐI ƯU CHO ẢNH NỀN ÂM TÍNH
class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        inter = (probs * targets).sum()
        dice = (2.0 * inter + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 0.5 * bce_loss + 0.5 * (1.0 - dice)

# 5. KHỞI TẠO DATALOADER, MÔ HÌNH VÀ OPTIMIZER
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = UNet(in_ch=3, out_ch=1).to(DEVICE)
criterion = BCEDiceLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# 6. VÒNG LẶP HUẤN LUYỆN
print(f" Bắt đầu huấn luyện UNet trên {DEVICE} (Seed={SEED}, Epochs={EPOCHS})...")
best_val_dice = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    
    scheduler.step()
    train_loss /= len(train_ds)
    
    # Đánh giá trên tập Validation
    model.eval()
    val_loss = 0.0
    dice_scores, iou_scores = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            val_loss += criterion(preds, masks).item() * imgs.size(0)
            
            probs = torch.sigmoid(preds) > 0.5
            for p, g in zip(probs, masks):
                inter = (p & (g > 0.5)).sum().float().item()
                union = (p | (g > 0.5)).sum().float().item()
                total = p.sum().float().item() + g.sum().float().item()
                
                # Xử lý trường hợp ảnh nền âm tính (g.sum() == 0)
                if g.sum().item() == 0:
                    d = 1.0 if p.sum().item() == 0 else 0.0
                    iou = 1.0 if p.sum().item() == 0 else 0.0
                else:
                    d = (2.0 * inter) / (total + 1e-5)
                    iou = inter / (union + 1e-5)
                dice_scores.append(d)
                iou_scores.append(iou)
                
    val_loss /= len(val_ds)
    val_dice = np.mean(dice_scores)
    val_iou = np.mean(iou_scores)
    
    # Lưu checkpoint tốt nhất
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f"best_unet_s{SEED}.pth"))
        saved_mark = " (BEST SAVED)"
    else:
        saved_mark = ""
        
    if epoch % 5 == 0 or epoch == 1 or saved_mark != "":
        print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}{saved_mark}")

print(f"\n Huấn luyện hoàn tất! Best Val Dice đạt: {best_val_dice:.4f}")
print(f"File trọng số lưu tại: {OUTPUT_DIR}/best_unet_s{SEED}.pth")

In [ ]:
!zip -r /kaggle/working/runs.zip /kaggle/working/runs